# Partie 2 — Traitement batch avec SparkCore (RDD)

**Objectif** : manipuler l'API bas niveau (RDD) sur l'historique de commandes.

| Élément | Valeur |
|---|---|
| Source | `s3a://lake/bronze/history/orders_history.jsonl` (couche Bronze, MinIO) |
| Volume | 150 281 lignes JSON — 50 000 commandes (`producer.py --mode file --n 50000`, seed 42) |
| Cluster | Spark 3.5.9 standalone, 2 workers × 4 cœurs, 4 executors × 2 cœurs |

> ⚠️ Données issues de la **reconstitution** de `producer.py` (fichier du professeur non fourni).

**Point d'attention métier** : le générateur injecte ~2 % de **doublons** (même `order_id`, même `status`).
Additionner naïvement les `DELIVERED` compterait certaines commandes deux fois. On dédoublonne donc par
`order_id` (avec `reduceByKey`, toujours en RDD) et on mesure l'écart avec le calcul naïf.

Toutes les opérations métier utilisent **exclusivement l'API RDD** (aucun DataFrame).

In [1]:
import json
import socket
import statistics
import time
import urllib.request
from operator import add
from pathlib import Path

from common.spark_session import get_spark, lake_path

spark = get_spark("part2-rdd", cores_max=8)   # tout le cluster : 8 cœurs
sc = spark.sparkContext
sc.setLogLevel("WARN")

HISTORY = lake_path("bronze", "history", "orders_history.jsonl")
RESTAURANTS = "file:///workspace/data/restaurants.json"   # même chemin sur driver et executors
LOCAL_HISTORY = "/workspace/data/generated/orders_history.jsonl"
LOGS = Path("/workspace/docs/logs")

METRICS = {"app_id": sc.applicationId, "spark_version": sc.version, "master": sc.master,
           "default_parallelism": sc.defaultParallelism, "timings_s": {}}


def timed(label, action):
    # Mesure le temps d'une ACTION (c'est elle qui déclenche réellement le calcul)
    t0 = time.perf_counter()
    result = action()
    dt = time.perf_counter() - t0
    METRICS["timings_s"][label] = round(dt, 3)
    print(f"⏱  {label:<40} {dt:8.3f} s")
    return result


# Attendre l'enregistrement des 4 executors (2 par worker)
deadline = time.time() + 60
while sc._jsc.sc().getExecutorMemoryStatus().size() - 1 < 4 and time.time() < deadline:
    time.sleep(1)

print(f"Spark {sc.version} | master={sc.master} | app={sc.appName} ({sc.applicationId})")
print(f"defaultParallelism={sc.defaultParallelism} | executors={sc._jsc.sc().getExecutorMemoryStatus().size() - 1}")
print(f"Spark UI : {sc.uiWebUrl}  (depuis l'hôte : http://localhost:4040)")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/14 17:36:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.9 | master=spark://spark-master:7077 | app=part2-rdd (app-20261114173614-0002)
defaultParallelism=8 | executors=4
Spark UI : http://jupyter:4040  (depuis l'hôte : http://localhost:4040)


## Étape 1 — Chargement en RDD et parsing JSON

- `sc.textFile()` crée un RDD de lignes : c'est une **transformation paresseuse**, rien n'est lu à ce stade.
- `minPartitions=8` : le fichier (32 Mio) tiendrait sinon dans **2 partitions** seulement
  (`min(defaultParallelism, 2)`), soit 2 tâches pour 8 cœurs disponibles.
- Le parsing utilise `flatMap` : une ligne invalide produit une liste vide au lieu de faire échouer le job.

In [2]:
def parse(line):
    try:
        return [json.loads(line)]
    except ValueError:
        return []          # ligne corrompue ignorée (comptée ci-dessous)


raw = sc.textFile(HISTORY, minPartitions=8)        # transformation (lazy)
events = raw.flatMap(parse)                         # transformation (lazy)

sc.setJobDescription("P2-1 lecture + parsing")
n_raw = timed("1_count_lignes_brutes", raw.count)            # ACTION
n_events = timed("1_count_evenements_parses", events.count)  # ACTION (relit S3 : pas de cache)

print(f"\nPartitions du RDD      : {raw.getNumPartitions()}")
print(f"Lignes brutes          : {n_raw}")
print(f"Événements parsés      : {n_events}")
print(f"Lignes invalides       : {n_raw - n_events}")
print("\nPremier événement      :", events.first())               # ACTION
METRICS.update(partitions=raw.getNumPartitions(), lines=n_raw, events=n_events)

26/11/14 17:36:22 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


⏱  1_count_lignes_brutes                       9.297 s


⏱  1_count_evenements_parses                   0.959 s

Partitions du RDD      : 8
Lignes brutes          : 150281
Événements parsés      : 150281
Lignes invalides       : 0



Premier événement      : {'order_id': 'ORD-0000001', 'restaurant_id': 'R-107', 'customer_id': 'C-10039', 'city': 'Marrakech', 'status': 'CREATED', 'amount_mad': 114.41, 'items_count': 2, 'distance_km': 12.6, 'courier_id': 'L-MAR-05', 'event_time': '2026-09-30T23:00:23Z'}


## Étape 2 — `filter` : commandes `DELIVERED` (+ dédoublonnage)

`filter` est une transformation **étroite** (*narrow*) : chaque partition est filtrée localement, sans échange réseau.
Le dédoublonnage par `order_id` utilise `reduceByKey(lambda a, b: a)` : une seule occurrence est gardée par clé.
Le RDD obtenu est mis en **cache** car il est réutilisé par les étapes 3, 4 et 5 (sans cache, chaque action relirait MinIO).

In [3]:
delivered = events.filter(lambda e: e["status"] == "DELIVERED")          # transformation étroite
delivered_unique = (delivered
                    .map(lambda e: (e["order_id"], e))                    # clé = order_id
                    .reduceByKey(lambda a, b: a)                          # 1 occurrence / commande (shuffle)
                    .values())
delivered_unique.cache()

sc.setJobDescription("P2-2 filtre DELIVERED + dédoublonnage + cache")
n_delivered_raw = timed("2_count_delivered_brut", delivered.count)
n_delivered = timed("2_count_delivered_dedup_mise_en_cache", delivered_unique.count)
n_delivered_cached = timed("2_count_delivered_depuis_cache", delivered_unique.count)

print(f"\nDELIVERED (brut, avec doublons) : {n_delivered_raw}")
print(f"DELIVERED (commandes uniques)   : {n_delivered}")
print(f"Doublons DELIVERED écartés      : {n_delivered_raw - n_delivered}")
METRICS.update(delivered_raw=n_delivered_raw, delivered_unique=n_delivered)

⏱  2_count_delivered_brut                      0.799 s


⏱  2_count_delivered_dedup_mise_en_cache       3.181 s


⏱  2_count_delivered_depuis_cache              0.428 s

DELIVERED (brut, avec doublons) : 46977
DELIVERED (commandes uniques)   : 46028
Doublons DELIVERED écartés      : 949


## Étape 3 — Chiffre d'affaires par ville (`map` + `reduceByKey`)

`map` produit des paires `(ville, montant)` ; `reduceByKey(add)` additionne par clé.
`reduceByKey` provoque un **shuffle** (transformation *large*) : les paires d'une même ville doivent être
regroupées sur la même partition. Spark pré-agrège d'abord **localement dans chaque partition**
(*map-side combine*), ce qui réduit fortement le volume échangé : au plus 8 villes × 8 partitions transitent.

In [4]:
ca_by_city_rdd = (delivered_unique
                  .map(lambda e: (e["city"], e["amount_mad"]))   # transformation étroite
                  .reduceByKey(add))                             # transformation large (shuffle)

sc.setJobDescription("P2-3 CA par ville (map + reduceByKey)")
ca_by_city = timed("3_ca_par_ville", lambda: sorted(ca_by_city_rdd.collect(), key=lambda kv: -kv[1]))
total_ca = sum(v for _, v in ca_by_city)

# Calcul naïf (sans dédoublonnage) pour mesurer l'impact des doublons
sc.setJobDescription("P2-3b CA naïf (sans dédoublonnage)")
ca_naive = dict(delivered.map(lambda e: (e["city"], e["amount_mad"])).reduceByKey(add).collect())

print(f"\n{'Ville':<12}{'CA (MAD)':>16}{'Part':>8}{'CA naïf (MAD)':>18}{'Écart':>12}")
print("-" * 66)
for city, ca in ca_by_city:
    print(f"{city:<12}{ca:>16,.2f}{ca / total_ca:>8.1%}{ca_naive[city]:>18,.2f}{ca_naive[city] - ca:>12,.2f}")
print("-" * 66)
print(f"{'TOTAL':<12}{total_ca:>16,.2f}{1:>8.0%}{sum(ca_naive.values()):>18,.2f}{sum(ca_naive.values()) - total_ca:>12,.2f}")

⏱  3_ca_par_ville                              1.184 s



Ville               CA (MAD)    Part     CA naïf (MAD)       Écart
------------------------------------------------------------------
Casablanca      2,441,036.72   27.7%      2,488,993.49   47,956.77
Marrakech       1,252,086.80   14.2%      1,275,235.66   23,148.86
Agadir          1,066,986.41   12.1%      1,089,938.34   22,951.93
Rabat           1,003,432.78   11.4%      1,025,606.34   22,173.56
Tanger            953,252.19   10.8%        976,258.05   23,005.86
Fes               872,019.55    9.9%        888,223.34   16,203.79
Oujda             833,093.75    9.5%        851,804.02   18,710.27
Meknes            377,737.33    4.3%        386,011.88    8,274.55
------------------------------------------------------------------
TOTAL           8,799,645.53    100%      8,982,071.12  182,425.59


## Étape 4 — Panier moyen par ville (`aggregateByKey`, un seul passage)

`aggregateByKey(zero, seqOp, combOp)` calcule **somme et nombre en un seul passage** :
- `zero = (0.0, 0)` : accumulateur initial par clé et par partition ;
- `seqOp` : intègre un montant dans l'accumulateur **à l'intérieur d'une partition** ;
- `combOp` : fusionne les accumulateurs **de partitions différentes** (après le shuffle).

Le panier moyen est ensuite `somme / nombre` (`mapValues`, transformation étroite qui conserve le partitionnement).

In [5]:
zero = (0.0, 0)


def seq_op(acc, amount):          # dans une partition
    return (acc[0] + amount, acc[1] + 1)


def comb_op(a, b):                # entre partitions
    return (a[0] + b[0], a[1] + b[1])


sum_count_rdd = (delivered_unique
                 .map(lambda e: (e["city"], e["amount_mad"]))
                 .aggregateByKey(zero, seq_op, comb_op))
avg_basket_rdd = sum_count_rdd.mapValues(lambda s: (s[0], s[1], s[0] / s[1]))

sc.setJobDescription("P2-4 panier moyen (aggregateByKey)")
avg_basket = timed("4_panier_moyen_aggregateByKey", lambda: sorted(avg_basket_rdd.collect(), key=lambda kv: -kv[1][2]))

print(f"\n{'Ville':<12}{'Commandes':>11}{'CA (MAD)':>16}{'Panier moyen (MAD)':>21}")
print("-" * 60)
for city, (ca, n, avg) in avg_basket:
    print(f"{city:<12}{n:>11,}{ca:>16,.2f}{avg:>21.2f}")
n_all = sum(v[1] for _, v in avg_basket)
print("-" * 60)
print(f"{'GLOBAL':<12}{n_all:>11,}{total_ca:>16,.2f}{total_ca / n_all:>21.2f}")

# Contrôle de cohérence : même CA qu'à l'étape 3, même nombre de commandes qu'à l'étape 2
ca_dict = dict(ca_by_city)
assert all(abs(ca_dict[c] - v[0]) < 1e-6 for c, v in avg_basket), "CA incohérent entre étapes 3 et 4"
assert n_all == n_delivered, "nombre de commandes incohérent"
print("\n✔ Cohérence : CA identique à l'étape 3 et", n_all, "commandes = DELIVERED uniques")

⏱  4_panier_moyen_aggregateByKey               1.236 s

Ville         Commandes        CA (MAD)   Panier moyen (MAD)
------------------------------------------------------------
Agadir            5,407    1,066,986.41               197.33
Marrakech         6,399    1,252,086.80               195.67
Tanger            4,893      953,252.19               194.82
Casablanca       12,592    2,441,036.72               193.86
Fes               4,599      872,019.55               189.61
Meknes            2,046      377,737.33               184.62
Rabat             5,498    1,003,432.78               182.51
Oujda             4,594      833,093.75               181.34
------------------------------------------------------------
GLOBAL           46,028    8,799,645.53               191.18

✔ Cohérence : CA identique à l'étape 3 et 46028 commandes = DELIVERED uniques


## Étape 5 — Top 5 des restaurants par commandes livrées (`takeOrdered`)

- `takeOrdered(5, key)` garde les 5 meilleurs **dans chaque partition** puis fusionne côté driver :
  pas de tri global, donc pas de shuffle supplémentaire.
- `sortBy(...).take(5)` (variante demandée) trie **tout** le RDD : shuffle complet + un job
  d'échantillonnage pour construire le `RangePartitioner`. On compare les deux.
- Les noms des restaurants viennent du référentiel, diffusé à chaque executor par une **variable broadcast**
  (envoyée une seule fois par executor au lieu d'une fois par tâche).

In [6]:
sc.setJobDescription("P2-5 chargement du référentiel (broadcast)")
restaurants = (sc.textFile(RESTAURANTS)
               .map(json.loads)
               .map(lambda r: (r["restaurant_id"], r))
               .collectAsMap())                                   # ACTION (200 lignes)
bc_rest = sc.broadcast(restaurants)

count_by_rest = delivered_unique.map(lambda e: (e["restaurant_id"], 1)).reduceByKey(add)
order_key = lambda kv: (-kv[1], kv[0])                            # nb décroissant, puis id (départage stable)

sc.setJobDescription("P2-5a top 5 restaurants (takeOrdered)")
top5 = timed("5_top5_takeOrdered", lambda: count_by_rest.takeOrdered(5, key=order_key))
sc.setJobDescription("P2-5b top 5 restaurants (sortBy + take)")
top5_sortby = timed("5_top5_sortBy_take", lambda: count_by_rest.sortBy(order_key).take(5))
assert top5 == top5_sortby, "takeOrdered et sortBy+take divergent"

# CA de ces restaurants (enrichissement via broadcast côté executors)
sc.setJobDescription("P2-5c CA des restaurants du top 5")
top_ids = {rid for rid, _ in top5}
ca_top = dict(delivered_unique.filter(lambda e: e["restaurant_id"] in top_ids)
              .map(lambda e: (e["restaurant_id"], e["amount_mad"])).reduceByKey(add).collect())

print(f"\n{'#':<3}{'ID':<7}{'Restaurant':<30}{'Ville':<12}{'Catégorie':<11}{'Livrées':>8}{'CA (MAD)':>14}")
print("-" * 85)
for rank, (rid, n) in enumerate(top5, 1):
    r = bc_rest.value[rid]
    print(f"{rank:<3}{rid:<7}{r['name']:<30}{r['city']:<12}{r['category']:<11}{n:>8}{ca_top[rid]:>14,.2f}")
print("\n✔ takeOrdered et sortBy + take donnent le même classement")

⏱  5_top5_takeOrdered                          0.971 s


⏱  5_top5_sortBy_take                          1.512 s



#  ID     Restaurant                    Ville       Catégorie   Livrées      CA (MAD)
-------------------------------------------------------------------------------------
1  R-157  Bab Fassia Dakhla             Agadir      marocain       1863    416,203.57
2  R-188  Forno Roma Lazaret            Oujda       pizza          1157    242,785.20
3  R-013  Shawarma Express Maarif       Casablanca  shawarma        904    106,559.56
4  R-147  Smash Spot Iberia             Tanger      burger          890    154,244.92
5  R-012  Forno Mamma Oasis             Casablanca  pizza           728    152,653.90

✔ takeOrdered et sortBy + take donnent le même classement


## Étape 6 — Lineage (`toDebugString`) : transformations vs actions

Le *lineage* est le graphe des transformations qui permet de **recalculer** une partition perdue.
Dans la sortie de `toDebugString()`, chaque **décalage d'indentation** (`+-`) marque une frontière de
**shuffle**, donc une frontière de **stage**. Les RDD marqués `CachedPartitions` sont servis depuis le cache.

In [7]:
def lineage(rdd):
    s = rdd.toDebugString()
    return s.decode("utf-8") if isinstance(s, bytes) else s


lineage_txt = []
for name, rdd in [("ca_by_city_rdd   (étape 3)", ca_by_city_rdd),
                  ("avg_basket_rdd   (étape 4)", avg_basket_rdd),
                  ("count_by_rest    (étape 5)", count_by_rest)]:
    block = f"===== {name}\n{lineage(rdd)}\n"
    lineage_txt.append(block)
    print(block)

(LOGS / "09_part2_lineage.txt").write_text("\n".join(lineage_txt), encoding="utf-8")

===== ca_by_city_rdd   (étape 3)
(8) PythonRDD[17] at collect at /tmp/ipykernel_1013/26165073.py:6 []
 |  MapPartitionsRDD[16] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[15] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(8) PairwiseRDD[14] at reduceByKey at /tmp/ipykernel_1013/26165073.py:3 []
    |  PythonRDD[13] at reduceByKey at /tmp/ipykernel_1013/26165073.py:3 []
    |  PythonRDD[9] at RDD at PythonRDD.scala:53 []
    |      CachedPartitions: 8; MemorySize: 2.4 MiB; DiskSize: 0.0 B
    |  MapPartitionsRDD[8] at mapPartitions at PythonRDD.scala:160 []
    |  ShuffledRDD[7] at partitionBy at NativeMethodAccessorImpl.java:0 []
    +-(8) PairwiseRDD[6] at reduceByKey at /tmp/ipykernel_1013/1229517039.py:4 []
       |  PythonRDD[5] at reduceByKey at /tmp/ipykernel_1013/1229517039.py:4 []
       |  s3a://lake/bronze/history/orders_history.jsonl MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0 []
       |  s3a://lake/bronze/history/orders_histo

3218

| Opération utilisée | Type | Étroite / large | Effet |
|---|---|---|---|
| `textFile` | transformation (lazy) | — | crée le RDD source (`HadoopRDD` → `MapPartitionsRDD`) |
| `flatMap`, `map`, `filter`, `values`, `mapValues` | transformation (lazy) | étroite | calcul local à chaque partition |
| `reduceByKey`, `aggregateByKey` | transformation (lazy) | **large** | shuffle → nouveau stage |
| `sortBy` | transformation (lazy)* | **large** | shuffle ; *lance un job d'échantillonnage (RangePartitioner) |
| `cache` | marqueur (lazy) | — | matérialisé à la 1re action |
| `count`, `first`, `collect`, `collectAsMap`, `take`, `takeOrdered` | **action** | — | déclenche un job, renvoie un résultat au driver |

## Étape 7 — Preuve de distribution : les deux workers exécutent les tâches

1. Chaque partition déclare le nom d'hôte de la machine qui la traite (`mapPartitionsWithIndex`).
2. L'API REST du driver (`/api/v1/applications/<id>/executors` et `/jobs`) donne les tâches exécutées
   par executor ainsi que le nombre de stages de chaque job.

In [8]:
sc.setJobDescription("P2-7 preuve de distribution")
per_partition = timed("7_hostname_par_partition", lambda: raw.mapPartitionsWithIndex(
    lambda i, it: [(i, socket.gethostname(), sum(1 for _ in it))]).collect())

print(f"{'Partition':>9}  {'Worker':<16}{'Lignes':>9}")
for i, host, n in sorted(per_partition):
    print(f"{i:>9}  {host:<16}{n:>9,}")
hosts = sorted({h for _, h, _ in per_partition})
print(f"\nWorkers ayant traité des partitions : {hosts}")

api = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}"
executors = json.load(urllib.request.urlopen(f"{api}/executors"))
tasks_by_host = {}
print(f"\n{'Executor':<10}{'Adresse':<22}{'Worker':<18}{'Cœurs':>6}{'Tâches':>8}{'Shuffle R (Ko)':>16}{'Shuffle W (Ko)':>16}")
for e in sorted(executors, key=lambda x: x["id"]):
    ip = e["hostPort"].split(":")[0]
    try:
        host = socket.gethostbyaddr(ip)[0].split(".")[0]
    except OSError:
        host = ip
    if e["id"] != "driver":
        tasks_by_host[host] = tasks_by_host.get(host, 0) + e["totalTasks"]
    print(f"{e['id']:<10}{e['hostPort']:<22}{host:<18}{e['totalCores']:>6}{e['totalTasks']:>8}"
          f"{e['totalShuffleRead'] / 1024:>16,.1f}{e['totalShuffleWrite'] / 1024:>16,.1f}")
print(f"\nTâches par worker : {tasks_by_host}")
METRICS["tasks_by_worker"] = tasks_by_host
METRICS["partition_hosts"] = {str(i): h for i, h, _ in per_partition}
assert len(tasks_by_host) == 2 and all(v > 0 for v in tasks_by_host.values()), "un worker n'a exécuté aucune tâche"
print("✔ Les deux workers ont exécuté des tâches")

⏱  7_hostname_par_partition                    0.490 s
Partition  Worker             Lignes
        0  spark-worker-1     18,782
        1  spark-worker-2     18,785
        2  spark-worker-2     18,789
        3  spark-worker-1     18,789
        4  spark-worker-1     18,787
        5  spark-worker-2     18,787
        6  spark-worker-2     18,780
        7  spark-worker-1     18,782

Workers ayant traité des partitions : ['spark-worker-1', 'spark-worker-2']



Executor  Adresse               Worker             Cœurs  Tâches  Shuffle R (Ko)  Shuffle W (Ko)
0         172.22.0.9:45681      sbd-spark-worker-2     2      42           661.9           653.9
1         172.22.0.9:36421      sbd-spark-worker-2     2      40           658.7           650.8
2         172.22.0.8:41109      sbd-spark-worker-1     2      42           664.2           646.7
3         172.22.0.8:40683      sbd-spark-worker-1     2      40           668.3           655.5
driver    jupyter:37127         jupyter                0       0             0.0             0.0

Tâches par worker : {'sbd-spark-worker-2': 82, 'sbd-spark-worker-1': 82}
✔ Les deux workers ont exécuté des tâches


In [9]:
jobs = json.load(urllib.request.urlopen(f"{api}/jobs"))
print(f"{'Job':>4}  {'Description':<48}{'Stages':>7}{'Ignorés':>8}{'Tâches':>7}  Statut")
METRICS["jobs"] = []
for j in sorted(jobs, key=lambda x: x["jobId"]):
    desc = (j.get("description") or j["name"])[:46]
    print(f"{j['jobId']:>4}  {desc:<48}{len(j['stageIds']):>7}{j['numSkippedStages']:>8}{j['numTasks']:>7}  {j['status']}")
    METRICS["jobs"].append({"job_id": j["jobId"], "description": desc, "stages": len(j["stageIds"]),
                            "skipped_stages": j["numSkippedStages"], "tasks": j["numTasks"],
                            "status": j["status"]})

 Job  Description                                      Stages Ignorés Tâches  Statut
   0  P2-1 lecture + parsing                                1       0      8  SUCCEEDED
   1  P2-1 lecture + parsing                                1       0      8  SUCCEEDED
   2  P2-1 lecture + parsing                                1       0      1  SUCCEEDED
   3  P2-2 filtre DELIVERED + dédoublonnage + cache         1       0      8  SUCCEEDED
   4  P2-2 filtre DELIVERED + dédoublonnage + cache         2       0     16  SUCCEEDED
   5  P2-2 filtre DELIVERED + dédoublonnage + cache         2       1     16  SUCCEEDED
   6  P2-3 CA par ville (map + reduceByKey)                 3       1     24  SUCCEEDED
   7  P2-3b CA naïf (sans dédoublonnage)                    2       0     16  SUCCEEDED
   8  P2-4 panier moyen (aggregateByKey)                    3       1     24  SUCCEEDED
   9  P2-5 chargement du référentiel (broadcast)            1       0      2  SUCCEEDED
  10  P2-5a top 5 restaurants (take

## Étape 8 — Benchmark de référence pour la comparaison avec les DataFrames (Partie 3)

Pipeline **complet et sans cache**, identique à celui qui sera refait en DataFrame :
lecture S3 → parsing → filtre `DELIVERED` → dédoublonnage par `order_id` → CA par ville.
1 exécution de chauffe (JIT, connexions S3) puis 5 exécutions mesurées ; on retient la **médiane**.

In [10]:
def rdd_ca_pipeline():
    return dict(sc.textFile(HISTORY, minPartitions=8)
                .flatMap(parse)
                .filter(lambda e: e["status"] == "DELIVERED")
                .map(lambda e: (e["order_id"], (e["city"], e["amount_mad"])))
                .reduceByKey(lambda a, b: a)
                .values()
                .reduceByKey(add)
                .collect())


sc.setJobDescription("P2-8 benchmark CA par ville (chauffe)")
rdd_ca_pipeline()
runs = []
for i in range(5):
    sc.setJobDescription(f"P2-8 benchmark CA par ville (run {i + 1})")
    t0 = time.perf_counter()
    bench_result = rdd_ca_pipeline()
    runs.append(round(time.perf_counter() - t0, 3))
assert all(abs(bench_result[c] - v) < 1e-6 for c, v in ca_dict.items())
METRICS["benchmark_ca_par_ville_rdd"] = {
    "description": "textFile S3 -> json.loads -> filter DELIVERED -> dedup order_id -> reduceByKey ville, sans cache",
    "runs_s": runs, "median_s": statistics.median(runs), "min_s": min(runs), "max_s": max(runs)}
print("Durées (s) :", runs)
print(f"Médiane    : {statistics.median(runs):.3f} s")

Durées (s) : [1.917, 1.578, 1.489, 1.729, 1.577]
Médiane    : 1.578 s


## Étape 9 — Contrôle indépendant (Python pur, hors Spark)

Les résultats Spark sont recalculés en Python standard, sur le fichier local identique (même SHA-256
que l'objet MinIO), pour valider les chiffres de façon indépendante.

In [11]:
t0 = time.perf_counter()
seen, ca_py, cnt_py, rest_py = set(), {}, {}, {}
with open(LOCAL_HISTORY, encoding="utf-8") as fh:
    for line in fh:
        e = json.loads(line)
        if e["status"] != "DELIVERED" or e["order_id"] in seen:
            continue
        seen.add(e["order_id"])
        ca_py[e["city"]] = ca_py.get(e["city"], 0.0) + e["amount_mad"]
        cnt_py[e["city"]] = cnt_py.get(e["city"], 0) + 1
        rest_py[e["restaurant_id"]] = rest_py.get(e["restaurant_id"], 0) + 1
python_s = time.perf_counter() - t0
top5_py = sorted(rest_py.items(), key=order_key)[:5]

ok_ca = all(abs(ca_py[c] - ca_dict[c]) < 0.01 for c in ca_dict) and set(ca_py) == set(ca_dict)
ok_avg = all(abs(ca_py[c] / cnt_py[c] - v[2]) < 0.01 and cnt_py[c] == v[1] for c, v in avg_basket)
ok_top = top5_py == top5
print(f"CA par ville identique      : {ok_ca}")
print(f"Panier moyen identique      : {ok_avg}")
print(f"Top 5 identique             : {ok_top}")
print(f"Durée Python mono-thread    : {python_s:.3f} s (fichier local, sans réseau)")
METRICS["controle_python"] = {"ca_ok": ok_ca, "avg_ok": ok_avg, "top5_ok": ok_top, "duree_s": round(python_s, 3)}
assert ok_ca and ok_avg and ok_top

CA par ville identique      : True
Panier moyen identique      : True
Top 5 identique             : True
Durée Python mono-thread    : 5.583 s (fichier local, sans réseau)


## Étape 10 — Sauvegarde des résultats et métriques

- `docs/logs/09_part2_results.json` : CA, panier moyen, top 5 ;
- `docs/logs/09_part2_metrics.json` : temps, partitions, jobs/stages, tâches par worker (comparaison Partie 3) ;
- `docs/logs/09_part2_lineage.txt` : sorties `toDebugString()`.

Le DAG reste consultable après l'arrêt dans le **Spark History Server** : http://localhost:18080

In [12]:
results = {
    "ca_par_ville_mad": {c: round(v, 2) for c, v in ca_by_city},
    "ca_total_mad": round(total_ca, 2),
    "ca_naif_sans_dedup_mad": {c: round(v, 2) for c, v in ca_naive.items()},
    "panier_moyen_par_ville": {c: {"commandes": n, "ca_mad": round(ca, 2), "panier_moyen_mad": round(avg, 2)}
                               for c, (ca, n, avg) in avg_basket},
    "top5_restaurants": [{"rang": i, "restaurant_id": rid, "name": bc_rest.value[rid]["name"],
                          "city": bc_rest.value[rid]["city"], "category": bc_rest.value[rid]["category"],
                          "commandes_livrees": n, "ca_mad": round(ca_top[rid], 2)}
                         for i, (rid, n) in enumerate(top5, 1)],
}
(LOGS / "09_part2_results.json").write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
(LOGS / "09_part2_metrics.json").write_text(json.dumps(METRICS, indent=2, ensure_ascii=False), encoding="utf-8")
print("Temps mesurés (s) :")
for k, v in METRICS["timings_s"].items():
    print(f"  {k:<42}{v:>8.3f}")
print(f"  {'benchmark CA (médiane sur 5)':<42}{METRICS['benchmark_ca_par_ville_rdd']['median_s']:>8.3f}")

Temps mesurés (s) :
  1_count_lignes_brutes                        9.297
  1_count_evenements_parses                    0.959
  2_count_delivered_brut                       0.799
  2_count_delivered_dedup_mise_en_cache        3.181
  2_count_delivered_depuis_cache               0.428
  3_ca_par_ville                               1.184
  4_panier_moyen_aggregateByKey                1.236
  5_top5_takeOrdered                           0.971
  5_top5_sortBy_take                           1.512
  7_hostname_par_partition                     0.490
  benchmark CA (médiane sur 5)                 1.578


In [13]:
delivered_unique.unpersist()
bc_rest.destroy()
spark.stop()
print("SparkSession arrêtée — DAG disponibles dans le History Server (http://localhost:18080)")

SparkSession arrêtée — DAG disponibles dans le History Server (http://localhost:18080)
